# Map 3 — Home-Based Business Density

**Project:** Suburban Futures (MDAP, University of Melbourne)  
**Purpose:** Build a Greater Melbourne SA2 map of non-employing businesses per 1,000 working-age residents, with a 2019-2025 time slider and a 2025 static preview.

Workflow:

1. Inspect the ABS source workbooks.
2. Build the tidy Map 3 CSV and sanity log from CABEE and ERP inputs.
3. Build the interactive Plotly HTML map and static 2025 PNG preview.

Local data and generated outputs live outside Git, usually in the project Mediaflux collection.

## Inspect Source Files — Setup

This setup cell locates the downloaded ABS workbooks and defines helper functions for inspecting raw sheet structure. This inspection section is intentionally read-only: it prints workbook metadata, likely header rows, SA2 naming examples, and methodology clues without writing any processed data.

In [ ]:

from pathlib import Path
import re
import traceback

import pandas as pd
from openpyxl import load_workbook

CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not find the repo root containing the data/ directory.")

FILES = {
    'CABEE 2021-2025': REPO_ROOT / 'data' / 'raw' / 'abs' / 'cabee' / 'cabee_sa2_industry_employment_size_2021_2025.xlsx',
    'CABEE 2018-2022': REPO_ROOT / 'data' / 'raw' / 'abs' / 'cabee' / 'cabee_sa2_industry_employment_size_2018_2022.xlsx',
    'ERP by SA2 age/sex 2001-2024': REPO_ROOT / 'data' / 'raw' / 'abs' / 'population_age_sex' / 'erp_sa2_age_sex_2001_2024.xlsx',
}
SEARCH_TERMS = [
    'Tarneit', 'Cranbourne', 'Wyndham', 'Melton', 'Mickleham',
    'Melbourne CBD', 'Southbank', 'Docklands'
]

def size_mb(path):
    return path.stat().st_size / (1024 * 1024)

def workbook_sheet_dimensions(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    dims = []
    for ws in wb.worksheets:
        dims.append((ws.title, ws.max_row, ws.max_column))
    wb.close()
    return dims

def print_file_overview(path):
    print(f'Full path: {path}')
    print(f'File size: {size_mb(path):.2f} MB')
    print('Sheet names and dimensions:')
    for sheet, rows, cols in workbook_sheet_dimensions(path):
        print(f'  - {sheet}: {rows:,} rows x {cols:,} columns')

def print_raw_rows(path, sheet_name, n=15):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=n, dtype=object, engine='openpyxl')
    print(f'First {n} raw rows from sheet {sheet_name!r}:')
    print(raw.to_string(index=True, header=False))

def flatten_columns(columns):
    flattened = []
    for col in columns:
        if isinstance(col, tuple):
            parts = [str(x).strip() for x in col if pd.notna(x) and not str(x).startswith('Unnamed')]
            if len(parts) >= 2 and parts[-1].lower() in {'no.', 'no', 'number'}:
                flattened.append(parts[0])
            elif parts:
                flattened.append(' '.join(parts))
            else:
                flattened.append('')
        else:
            flattened.append(str(col).strip())
    return flattened

def load_cabee_sheet(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, header=[4, 5], dtype=object, engine='openpyxl')
    df.columns = flatten_columns(df.columns)
    return df

def load_erp_sheet(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, header=6, dtype=object, engine='openpyxl')
    df.columns = [str(c).strip() for c in df.columns]
    return df

def clean_cabee_data(df):
    out = df.copy()
    out = out[pd.to_numeric(out['SA2 Code'], errors='coerce').notna()].copy()
    return out

def clean_erp_data(df):
    out = df.copy()
    out = out[pd.to_numeric(out['SA2 code'], errors='coerce').notna()].copy()
    return out

def print_loaded_df_details(df, label):
    print(f'{label} loaded dataframe column names:')
    print(list(df.columns))
    print('\nDtypes:')
    print(df.dtypes.to_string())
    print(f'\nLoaded dataframe row count: {len(df):,}')
    print('\nhead(5):')
    print(df.head(5).to_string(index=False))

def print_sa2_samples(df, sa2_col):
    mask = pd.Series(False, index=df.index)
    for term in SEARCH_TERMS:
        mask = mask | df[sa2_col].astype(str).str.contains(term, case=False, na=False, regex=False)
    samples = sorted(df.loc[mask, sa2_col].dropna().astype(str).unique())[:10]
    print(f'SA2 sample names containing search terms ({sa2_col}):')
    if samples:
        for name in samples:
            print(f'  - {name}')
    else:
        print('  No matches found.')

def cabee_years_from_sheets(path):
    years = []
    wb = load_workbook(path, read_only=True, data_only=True)
    for ws in wb.worksheets:
        # CABEE data sheets have the data year in row 4 table titles.
        # This avoids picking up release-range years from the workbook banner.
        title_parts = []
        for row in ws.iter_rows(min_row=4, max_row=4, values_only=True):
            title_parts.extend(str(v) for v in row if v is not None)
        title = ' '.join(title_parts)
        if 'Businesses by Industry Division by Statistical Area Level 2' in title:
            years.extend(re.findall(r'June (20\d{2})', title))
    wb.close()
    return sorted(set(years))

def choose_sa2(df):
    names = df['SA2 Label'].dropna().astype(str)
    if (names == 'Tarneit - North').any():
        return 'Tarneit - North'
    wyndham = names[names.str.contains('Wyndham', case=False, na=False)]
    if not wyndham.empty:
        return sorted(wyndham.unique())[0]
    return sorted(names.unique())[0]

def inspect_cabee(path, release_label, likely_sheet):
    print_file_overview(path)
    print()
    print(f'Most likely SA2-level data sheet: {likely_sheet!r}')
    print('Actual column header rows: Excel rows 5-6; pandas header=[4, 5].')
    print_raw_rows(path, likely_sheet)
    print()
    raw_loaded = load_cabee_sheet(path, likely_sheet)
    data = clean_cabee_data(raw_loaded)
    print_loaded_df_details(data, f'{release_label} / {likely_sheet}')
    print(f'Raw rows after header before dropping footer/non-SA2 rows: {len(raw_loaded):,}')
    print(f'Clean SA2 data rows after dropping footer/non-SA2 rows: {len(data):,}')
    print()
    print('SA2 code column: SA2 Code')
    print('SA2 name column: SA2 Label')
    print_sa2_samples(data, 'SA2 Label')
    print()
    employment_cols = [c for c in data.columns if c not in ['Industry Code', 'Industry Label', 'SA2 Code', 'SA2 Label']]
    print('Distinct employment-size-range categories present:')
    for col in employment_cols:
        print(f'  - {col}')
    print('\nDistinct industry division values:')
    for value in sorted(data['Industry Label'].dropna().astype(str).unique()):
        print(f'  - {value}')
    years = cabee_years_from_sheets(path)
    print('\nYear columns present: none. Years are encoded in sheet names/table titles.')
    print(f'Data years detected in table sheets: {years}')
    chosen = choose_sa2(data)
    print(f'\nGreater Melbourne SA2 selected for shape check: {chosen}')
    sample_rows = data[data['SA2 Label'].eq(chosen)].copy()
    print(f'All rows for {chosen!r} in {likely_sheet!r}:')
    print(sample_rows.to_string(index=False))
    return chosen

def inspect_erp(path, likely_sheet, chosen_sa2='Tarneit - North'):
    print_file_overview(path)
    print()
    print(f'Most likely SA2-level data sheet: {likely_sheet!r}')
    print('Actual column header row: Excel row 7; pandas header=6.')
    print_raw_rows(path, likely_sheet)
    print()
    raw_loaded = load_erp_sheet(path, likely_sheet)
    data = clean_erp_data(raw_loaded)
    print_loaded_df_details(data, f'ERP / {likely_sheet}')
    print(f'Raw rows after header before dropping footer/non-SA2 rows: {len(raw_loaded):,}')
    print(f'Clean SA2 data rows after dropping footer/non-SA2 rows: {len(data):,}')
    print()
    print('SA2 code column: SA2 code')
    print('SA2 name column: SA2 name')
    print_sa2_samples(data, 'SA2 name')
    print()
    core_cols = ['Year', 'S/T code', 'S/T name', 'GCCSA code', 'GCCSA name', 'SA4 code', 'SA4 name', 'SA3 code', 'SA3 name', 'SA2 code', 'SA2 name']
    age_cols = [c for c in data.columns if c not in core_cols]
    print('Age bands present:')
    for col in age_cols:
        print(f'  - {col}')
    years = [int(y) for y in sorted(pd.to_numeric(data['Year'], errors='coerce').dropna().astype(int).unique())]
    print(f'\nYears present: {years}')
    latest = max(years)
    print(f'\nSample for {chosen_sa2!r} in latest year {latest}, all age bands:')
    sample = data[(data['SA2 name'].eq(chosen_sa2)) & (pd.to_numeric(data['Year'], errors='coerce').eq(latest))]
    if sample.empty:
        print(f'No exact row for {chosen_sa2!r}; showing first Tarneit-like row for {latest}.')
        sample = data[(data['SA2 name'].astype(str).str.contains('Tarneit', case=False, na=False)) & (pd.to_numeric(data['Year'], errors='coerce').eq(latest))].head(1)
    if sample.empty:
        print('No comparable sample found.')
    else:
        row = sample.iloc[0]
        print(f"SA2 code: {row['SA2 code']}")
        print(f"SA2 name: {row['SA2 name']}")
        print(f"Year: {row['Year']}")
        print(pd.DataFrame({'age_band': age_cols, 'population': [row[c] for c in age_cols]}).to_string(index=False))

def extract_older_cabee_asgs_sentence(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    direct_hits = []
    geography_hits = []
    for ws in wb.worksheets:
        for row_idx, row in enumerate(ws.iter_rows(values_only=True), start=1):
            text = ' '.join(str(v) for v in row if v is not None)
            if not text.strip():
                continue
            lower = text.lower()
            if 'asgs' in lower or 'australian statistical geography standard' in lower or 'edition' in lower:
                direct_hits.append((ws.title, row_idx, text))
            if 'classified to a single geography' in lower or 'smaller geographies, such as sa2 and lga' in lower:
                geography_hits.append((ws.title, row_idx, text))
    wb.close()
    print('ASGS edition sentence search in CABEE 2018-2022 workbook:')
    if direct_hits:
        for sheet, row_idx, text in direct_hits:
            print(f'  {sheet} row {row_idx}: {text}')
    else:
        print('  No explicit ASGS / Australian Statistical Geography Standard / Edition sentence found in the workbook text.')
        if geography_hits:
            sheet, row_idx, text = geography_hits[0]
            print('  Closest relevant geography methodology sentence found verbatim:')
            print(f'  {sheet} row {row_idx}: {text}')

missing = [str(path) for path in FILES.values() if not path.exists()]
if missing:
    print('Missing source file(s):')
    for path in missing:
        print(f'  - {path}')
    raise FileNotFoundError('Missing source file(s); stop inspection.')
print('Resolved source files:')
for label, path in FILES.items():
    print(f'  - {label}: {path} ({size_mb(path):.2f} MB)')


## File 1 — CABEE 2021-2025

Inspect the newest CABEE workbook first because it supplies the 2023-2025 frames. The important checks are the multi-row header location, the exact `Non employing` column label, the industry categories, and SA2 names used for later matching.

In [ ]:
try:
    SELECTED_SA2 = inspect_cabee(FILES['CABEE 2021-2025'], 'CABEE 2021-2025', 'Table 1')
except Exception:
    traceback.print_exc()


## File 2 — CABEE 2018-2022

Inspect the middle CABEE release because it supplies the 2020-2022 annualised `a` sheets. The methodology search at the end looks for geography/ASGS clues that would indicate whether a boundary correspondence step is needed.

In [ ]:
try:
    SELECTED_SA2_OLD = inspect_cabee(FILES['CABEE 2018-2022'], 'CABEE 2018-2022', '2022 a')
    print()
    extract_older_cabee_asgs_sentence(FILES['CABEE 2018-2022'])
except Exception:
    traceback.print_exc()


## File 3 — ERP by SA2 by Age and Sex 2001-2024

Inspect the ERP age/sex workbook used for the working-age denominator. The pipeline later sums ages 15-64 and carries 2024 population forward for the 2025 CABEE frame because ERP currently ends at 2024.

In [ ]:
try:
    inspect_erp(FILES['ERP by SA2 age/sex 2001-2024'], 'Table 3', globals().get('SELECTED_SA2', 'Tarneit - North'))
except Exception:
    traceback.print_exc()


## Build Analysis Table

This section builds the tidy Map 3 analysis table and a text sanity log. It does not create any visualisation, GeoJSON, Plotly, or HTML outputs.

Core methodology decisions:

- Use CABEE annualised `a` sheets only for 2019-2025.
- Use `Non employing` businesses as the numerator.
- Drop CABEE `Industry Code == "X"` because those rows are currently unknown state-level dumps.
- Use working-age residents aged 15-64 as the denominator.
- Apply a working-age population floor of 500 for map inclusion.
- Keep the seven CBD/Southbank/industrial-estate SA2s in the output but flag them for the alternate view.
- Use 2024 ERP population as the 2025 denominator because ERP currently ends at 2024.

Outputs from this section:

- `data/processed/map3_home_business_density.csv`
- `data/processed/map3_home_business_pipeline_log.txt`

### Configuration and Helper Functions

This cell defines all locked analysis-table parameters in one place: repo-relative source folders, year range, working-age bands, the 500-person denominator floor, CBD/distortion SA2 flags, output paths, expected raw-column schemas, and helper functions for loading CABEE and ERP files. It resolves the repo root from either the project folder or the `notebooks/` folder, so other developers do not need to edit absolute paths.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from openpyxl import load_workbook

CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not find the repo root containing the data/ directory.")

DATA_RAW_ABS_DIR = REPO_ROOT / "data" / "raw" / "abs"
CABEE_DIR = DATA_RAW_ABS_DIR / "cabee"
ERP_DIR = DATA_RAW_ABS_DIR / "population_age_sex"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

YEAR_RANGE = list(range(2019, 2026))
WORKING_AGE_BANDS = [
    "age_15_19", "age_20_24", "age_25_29", "age_30_34", "age_35_39",
    "age_40_44", "age_45_49", "age_50_54", "age_55_59", "age_60_64",
]
WORKING_AGE_FLOOR = 500

CBD_EXCLUSION_SA2_NAMES = [
    "Melbourne CBD - East",
    "Melbourne CBD - North",
    "Melbourne CBD - West",
    "Docklands",
    "Southbank - East",
    "Southbank (West) - South Wharf",
    "Port Melbourne Industrial",
]

CABEE_SOURCES = {
    2019: ("cabee_sa2_industry_employment_size_2017_2021.xlsx", "2019"),
    2020: ("cabee_sa2_industry_employment_size_2018_2022.xlsx", "2020 a"),
    2021: ("cabee_sa2_industry_employment_size_2018_2022.xlsx", "2021 a"),
    2022: ("cabee_sa2_industry_employment_size_2018_2022.xlsx", "2022 a"),
}
CABEE_2021_2025_FILE = "cabee_sa2_industry_employment_size_2021_2025.xlsx"
ERP_FILE = "erp_sa2_age_sex_2001_2024.xlsx"

MAP3_CSV_OUTPUT = PROCESSED_DIR / "map3_home_business_density.csv"
MAP3_LOG_OUTPUT = PROCESSED_DIR / "map3_home_business_pipeline_log.txt"

CABEE_COLUMNS = [
    "Industry Code", "Industry Label", "SA2 Code", "SA2 Label",
    "Non employing", "1-4 Employees", "5-19 Employees",
    "20-199 Employees", "200+ Employees", "Total",
]
ERP_COLUMNS = [
    "year", "st_code", "st_name",
    "gccsa_code", "gccsa_name",
    "sa4_code", "sa4_name",
    "sa3_code", "sa3_name",
    "sa2_code", "sa2_name",
    "age_0_4", "age_5_9", "age_10_14", "age_15_19", "age_20_24",
    "age_25_29", "age_30_34", "age_35_39", "age_40_44", "age_45_49",
    "age_50_54", "age_55_59", "age_60_64", "age_65_69", "age_70_74",
    "age_75_79", "age_80_84", "age_85_plus", "total_persons",
]
FINAL_COLUMNS = [
    "year", "sa2_code", "sa2_name", "non_employing_count", "working_age_pop",
    "total_persons", "non_employing_per_1000_working_age", "change_abs_since_2019",
    "change_pct_since_2019", "is_cbd_excluded", "has_sufficient_population",
    "include_in_primary_view", "include_in_alternate_view",
]


def resolve_sheet_name(workbook_path, requested_sheet):
    wb = load_workbook(workbook_path, read_only=True, data_only=True)
    try:
        stripped = {sheet.strip(): sheet for sheet in wb.sheetnames}
    finally:
        wb.close()
    requested_key = str(requested_sheet).strip()
    if requested_key in stripped:
        return stripped[requested_key]
    available = ", ".join(sorted(stripped))
    raise ValueError(f"Sheet {requested_sheet!r} not found in {workbook_path.name}; available sheets: {available}")


def parse_cabee_2021_2025_mapping(workbook_path):
    resolved = {}
    wb = load_workbook(workbook_path, read_only=True, data_only=True)
    try:
        stripped = {sheet.strip(): sheet for sheet in wb.sheetnames}
        for requested in ["Table 1", "Table 2", "Table 3"]:
            actual = stripped.get(requested)
            if actual is None:
                raise ValueError(f"Missing expected sheet {requested!r} in {workbook_path.name}")
            title = " ".join(str(v) for v in next(wb[actual].iter_rows(min_row=4, max_row=4, values_only=True)) if v is not None)
            match = re.search(r"June (20\d{2})", title)
            if not match:
                raise ValueError(f"Could not parse year from {workbook_path.name}/{actual} title row: {title!r}")
            year = int(match.group(1))
            resolved[year] = actual
    finally:
        wb.close()
    expected = {2023, 2024, 2025}
    if set(resolved) != expected:
        raise ValueError(f"2021-2025 CABEE sheet parsing expected {sorted(expected)}, got {sorted(resolved)}")
    return resolved


def normalize_sa2_code(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype("string")


def load_cabee_sheet(file_name, sheet_name, year, *, emit=print):
    workbook_path = CABEE_DIR / file_name
    actual_sheet = resolve_sheet_name(workbook_path, sheet_name)
    raw = pd.read_excel(
        workbook_path,
        sheet_name=actual_sheet,
        header=[4, 5],
        engine="openpyxl",
    )
    raw = raw.iloc[:, :len(CABEE_COLUMNS)].copy()
    raw.columns = CABEE_COLUMNS
    raw = raw[pd.to_numeric(raw["SA2 Code"], errors="coerce").notna()].copy()
    raw["SA2 Code"] = normalize_sa2_code(raw["SA2 Code"])
    raw["Industry Code"] = raw["Industry Code"].astype("string")
    raw["Industry Label"] = raw["Industry Label"].astype("string")
    raw["SA2 Label"] = raw["SA2 Label"].astype("string")
    raw["Non employing"] = pd.to_numeric(raw["Non employing"], errors="coerce").astype("Int64")
    raw["year"] = int(year)
    clean = raw[["year", "SA2 Code", "SA2 Label", "Industry Code", "Industry Label", "Non employing"]].copy()
    emit(f"Loaded {year} from {file_name}/{actual_sheet}: {len(clean):,} rows, {clean['SA2 Code'].nunique():,} unique SA2s")
    return clean


def aggregate_non_employing(cabee_long):
    filtered = cabee_long[cabee_long["Industry Code"].ne("X")].copy()
    return (
        filtered.groupby(["year", "SA2 Code", "SA2 Label"], as_index=False)["Non employing"]
        .sum()
        .rename(columns={
            "SA2 Code": "sa2_code",
            "SA2 Label": "sa2_name",
            "Non employing": "non_employing_count",
        })
    )


def load_erp_working_age():
    erp_path = ERP_DIR / ERP_FILE
    erp = pd.read_excel(erp_path, sheet_name="Table 3", header=None, skiprows=7, engine="openpyxl")
    if erp.shape[1] != len(ERP_COLUMNS):
        raise ValueError(f"ERP Table 3 expected exactly {len(ERP_COLUMNS)} columns, got {erp.shape[1]}")
    erp.columns = ERP_COLUMNS
    erp = erp[pd.to_numeric(erp["sa2_code"], errors="coerce").notna()].copy()
    erp["year"] = pd.to_numeric(erp["year"], errors="coerce").astype("Int64")
    erp = erp[erp["year"].notna()].copy()
    erp["year"] = erp["year"].astype(int)
    erp["sa2_code"] = normalize_sa2_code(erp["sa2_code"])
    erp["sa2_name"] = erp["sa2_name"].astype("string")
    for col in [c for c in erp.columns if c.startswith("age_")] + ["total_persons"]:
        erp[col] = pd.to_numeric(erp[col], errors="coerce").astype("Int64")
    erp["working_age_pop"] = erp[WORKING_AGE_BANDS].sum(axis=1).astype("Int64")
    return erp


def format_df(frame, columns=None, max_rows=None):
    out = frame.copy()
    if columns is not None:
        out = out[columns]
    if max_rows is not None:
        out = out.head(max_rows)
    return out.to_string(index=False)

### Run Pipeline, QA Checks, and Write Outputs

This cell executes the full analysis-table build. It validates source files, resolves CABEE sheet names, loads all seven years, checks boundary stability, compares overlapping CABEE releases, aggregates non-employing businesses, joins the ERP denominator, computes per-1,000 and change-since-2019 metrics, applies view flags, writes the CSV, and writes a plain-text sanity log. The printed sections are intentionally verbose because they are the audit trail for stakeholder review.

In [ ]:
log_lines = []


def log(message=""):
    text = str(message)
    print(text)
    log_lines.append(text)


def log_section(title):
    log()
    log(f"=== {title} ===")


def abort_if_missing_sources():
    required = [
        CABEE_DIR / "cabee_sa2_industry_employment_size_2017_2021.xlsx",
        CABEE_DIR / "cabee_sa2_industry_employment_size_2018_2022.xlsx",
        CABEE_DIR / CABEE_2021_2025_FILE,
        ERP_DIR / ERP_FILE,
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        log("Missing source file(s):")
        for path in missing:
            log(f"  - {path}")
        raise FileNotFoundError("Missing source file(s); aborting Map 3 pipeline")


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
abort_if_missing_sources()

log_section("Source files")
for path in [
    CABEE_DIR / "cabee_sa2_industry_employment_size_2017_2021.xlsx",
    CABEE_DIR / "cabee_sa2_industry_employment_size_2018_2022.xlsx",
    CABEE_DIR / CABEE_2021_2025_FILE,
    ERP_DIR / ERP_FILE,
]:
    log(f"{path} ({path.stat().st_size / (1024 * 1024):.2f} MB)")

log_section("Resolved CABEE 2021-2025 annualised sheet mapping")
cabee_2021_2025_mapping = parse_cabee_2021_2025_mapping(CABEE_DIR / CABEE_2021_2025_FILE)
for year in sorted(cabee_2021_2025_mapping):
    log(f"{year}: {CABEE_2021_2025_FILE}/{cabee_2021_2025_mapping[year]}")

all_cabee_sources = dict(CABEE_SOURCES)
for year, sheet in cabee_2021_2025_mapping.items():
    all_cabee_sources[year] = (CABEE_2021_2025_FILE, sheet)
if sorted(all_cabee_sources) != YEAR_RANGE:
    raise ValueError(f"Expected CABEE sources for {YEAR_RANGE}, got {sorted(all_cabee_sources)}")

log_section("CABEE load")
cabee_frames = []
for year in YEAR_RANGE:
    file_name, sheet_name = all_cabee_sources[year]
    frame = load_cabee_sheet(file_name, sheet_name, year, emit=log)
    if frame.empty:
        raise ValueError(f"CABEE load for {year} produced zero rows; aborting")
    cabee_frames.append(frame)
cabee_long = pd.concat(cabee_frames, ignore_index=True)

log_section("Boundary stability check")
sa2_sets_by_year = {
    year: set(cabee_long.loc[cabee_long["year"].eq(year), "SA2 Code"].dropna().astype(str))
    for year in YEAR_RANGE
}
boundary_summary = pd.DataFrame(
    {"year": year, "unique_sa2_count": len(sa2_sets_by_year[year])}
    for year in YEAR_RANGE
)
log(boundary_summary.to_string(index=False))
sym_diff = sa2_sets_by_year[2019].symmetric_difference(sa2_sets_by_year[2025])
log(f"Symmetric difference between 2019 and 2025 SA2 code sets: {len(sym_diff):,}")
if len(sym_diff) > 50:
    raise ValueError("boundary correspondence required, halting before downstream steps")

log_section("Overlap QA for 2017-2021 vs 2018-2022 releases")
for overlap_year in [2020, 2021]:
    old = load_cabee_sheet(
        "cabee_sa2_industry_employment_size_2017_2021.xlsx",
        f"{overlap_year} a",
        overlap_year,
        emit=lambda _msg: None,
    )
    new = load_cabee_sheet(
        "cabee_sa2_industry_employment_size_2018_2022.xlsx",
        f"{overlap_year} a",
        overlap_year,
        emit=lambda _msg: None,
    )
    old_agg = aggregate_non_employing(old).rename(columns={"non_employing_count": "old_non_employing"})
    new_agg = aggregate_non_employing(new).rename(columns={"non_employing_count": "new_non_employing"})
    qa = old_agg[["sa2_code", "old_non_employing"]].merge(
        new_agg[["sa2_code", "new_non_employing"]],
        on="sa2_code",
        how="inner",
    )
    denominator = qa["new_non_employing"].replace(0, pd.NA)
    qa["abs_relative_diff"] = (qa["old_non_employing"] - qa["new_non_employing"]).abs() / denominator
    qa.loc[qa["new_non_employing"].eq(0) & qa["old_non_employing"].eq(0), "abs_relative_diff"] = 0
    median_abs_rel_diff = qa["abs_relative_diff"].replace([np.inf, -np.inf], pd.NA).dropna().median()
    outlier_count = int((qa["abs_relative_diff"] > 0.10).sum())
    log(f"{overlap_year}: median absolute relative difference = {median_abs_rel_diff:.2%}; SA2s >10% = {outlier_count:,} of {len(qa):,}")

log_section("Industry filter and aggregation")
cabee_filtered = cabee_long[cabee_long["Industry Code"].ne("X")].copy()
log(f"CABEE long rows before dropping Industry Code X: {len(cabee_long):,}")
log(f"CABEE long rows after dropping Industry Code X: {len(cabee_filtered):,}")
cabee_aggregated = aggregate_non_employing(cabee_long)
log(f"Aggregated CABEE rows: {len(cabee_aggregated):,}")
log(cabee_aggregated.groupby("year").size().rename("rows").to_string())
if (cabee_aggregated.groupby("year").size() == 0).any():
    raise ValueError("At least one year has zero CABEE rows after filtering; aborting")

log_section("ERP load and Greater Melbourne filter")
erp = load_erp_working_age()
greater_melbourne_erp = erp[erp["gccsa_name"].eq("Greater Melbourne")].copy()
greater_melbourne_sa2_codes = set(greater_melbourne_erp["sa2_code"].dropna().astype(str))
log(f"Resolved Greater Melbourne SA2 count: {len(greater_melbourne_sa2_codes):,}")
log("2025 working-age denominator uses 2024 ERP figures (most recent available)")

erp_target = greater_melbourne_erp[greater_melbourne_erp["year"].isin(range(2019, 2025))].copy()
erp_2025 = erp_target[erp_target["year"].eq(2024)].copy()
erp_2025["year"] = 2025
erp_for_join = pd.concat([erp_target, erp_2025], ignore_index=True)
erp_for_join = erp_for_join[["year", "sa2_code", "sa2_name", "working_age_pop", "total_persons"]].copy()
erp_for_join["sa2_code"] = erp_for_join["sa2_code"].astype("string")

erp_working_age = erp_for_join.copy()

cabee_aggregated = cabee_aggregated[cabee_aggregated["sa2_code"].astype(str).isin(greater_melbourne_sa2_codes)].copy()
cabee_aggregated["sa2_code"] = cabee_aggregated["sa2_code"].astype("string")
log(f"CABEE aggregated Greater Melbourne rows: {len(cabee_aggregated):,}")
log(cabee_aggregated.groupby("year").size().rename("rows").to_string())

log_section("Join and metric computation")
df = erp_for_join.merge(
    cabee_aggregated,
    on=["year", "sa2_code"],
    how="left",
    suffixes=("_erp", "_cabee"),
)
df["sa2_name"] = df["sa2_name_cabee"].combine_first(df["sa2_name_erp"])
df = df.drop(columns=["sa2_name_erp", "sa2_name_cabee"])
df["non_employing_count"] = df["non_employing_count"].astype("Int64")
df["working_age_pop"] = df["working_age_pop"].astype("Int64")
df["total_persons"] = df["total_persons"].astype("Int64")
df["non_employing_per_1000_working_age"] = np.where(
    df["working_age_pop"] > 0,
    (df["non_employing_count"] / df["working_age_pop"] * 1000).round(1),
    np.nan,
)
zero_denominator_rows = df["working_age_pop"].le(0).sum()
log(f"Rows with working_age_pop <= 0 set to NaN for per-1000/change metrics: {int(zero_denominator_rows):,}")

baseline = (
    df[df["year"].eq(2019)][["sa2_code", "non_employing_per_1000_working_age"]]
    .rename(columns={"non_employing_per_1000_working_age": "baseline_2019"})
)
df = df.merge(baseline, on="sa2_code", how="left")
missing_or_invalid_baseline = df.groupby("sa2_code")["baseline_2019"].first().isna()
log(f"SA2s without valid 2019 baseline metric: {int(missing_or_invalid_baseline.sum()):,}")
df["change_abs_since_2019"] = (
    df["non_employing_per_1000_working_age"] - df["baseline_2019"]
).round(1)
df["change_pct_since_2019"] = np.where(
    df["baseline_2019"] > 0,
    ((df["non_employing_per_1000_working_age"] / df["baseline_2019"] - 1) * 100).round(1),
    np.nan,
)

df["is_cbd_excluded"] = df["sa2_name"].isin(CBD_EXCLUSION_SA2_NAMES)
df["has_sufficient_population"] = df["working_age_pop"] >= WORKING_AGE_FLOOR
df["include_in_primary_view"] = (~df["is_cbd_excluded"]) & df["has_sufficient_population"]
df["include_in_alternate_view"] = df["has_sufficient_population"]

final_df = df[FINAL_COLUMNS].sort_values(["year", "sa2_name"]).reset_index(drop=True)

log_section("1. Total row count of final table")
log(f"Total rows: {len(final_df):,}")

log_section("2. Count of rows per year where include_in_primary_view is True")
log(final_df[final_df["include_in_primary_view"]].groupby("year").size().rename("primary_view_rows").to_string())

log_section("3. CBD-excluded SA2 names found")
cbd_found = sorted(final_df.loc[final_df["is_cbd_excluded"], "sa2_name"].dropna().unique())
for name in cbd_found:
    log(f"- {name}")
cbd_missing = sorted(set(CBD_EXCLUSION_SA2_NAMES) - set(cbd_found))
if cbd_missing:
    log("Missing CBD exclusion names:")
    for name in cbd_missing:
        log(f"- {name}")
if len(cbd_found) < 6:
    log(f"WARNING: only {len(cbd_found)} of 7 CBD exclusion SA2s found; continuing")

log_section("4. SA2/year rows below the working-age floor")
low_population = final_df[~final_df["has_sufficient_population"]].sort_values(["working_age_pop", "year", "sa2_name"])
log(format_df(low_population[["year", "sa2_name", "working_age_pop", "non_employing_count", "non_employing_per_1000_working_age"]]))

frame_2025_primary = final_df[(final_df["year"].eq(2025)) & (final_df["include_in_primary_view"])].copy()

metric_cols = [
    "sa2_name", "working_age_pop", "non_employing_count",
    "non_employing_per_1000_working_age", "change_abs_since_2019", "change_pct_since_2019",
]

log_section("5. Top 15 SA2s by non_employing_per_1000_working_age, 2025 primary view")
log(format_df(frame_2025_primary.nlargest(15, "non_employing_per_1000_working_age"), metric_cols))

log_section("6. Bottom 15 SA2s by non_employing_per_1000_working_age, 2025 primary view")
log(format_df(frame_2025_primary.nsmallest(15, "non_employing_per_1000_working_age"), metric_cols))

log_section("7. Top 15 SA2s by change_abs_since_2019, 2025 primary view")
log(format_df(frame_2025_primary.nlargest(15, "change_abs_since_2019"), metric_cols))

log_section("8. Top 15 SA2s by change_pct_since_2019, 2025 primary view, baseline >=10 per 1000")
frame_2025_with_baseline = df[(df["year"].eq(2025)) & (df["include_in_primary_view"]) & (df["baseline_2019"] >= 10)].copy()
log(format_df(frame_2025_with_baseline.nlargest(15, "change_pct_since_2019"), metric_cols + ["baseline_2019"]))

log_section("9. Tarneit - North all-year anchor")
tarneit = final_df[final_df["sa2_name"].eq("Tarneit - North")].sort_values("year")
log(format_df(tarneit[["year", "non_employing_count", "working_age_pop", "non_employing_per_1000_working_age", "change_abs_since_2019"]]))

log_section("10. Growth-corridor SA2 2019 vs 2025 comparison")
growth_requested = ["Tarneit - North", "Cranbourne East - North", "Mickleham - Yuroke"]
for name in growth_requested:
    if name in set(final_df["sa2_name"]):
        selected_name = name
    else:
        matches = sorted([candidate for candidate in final_df["sa2_name"].dropna().unique() if name.split(" - ")[0] in candidate])
        selected_name = matches[0] if matches else None
    if selected_name is None:
        log(f"No match found for {name}")
        continue
    comparison = final_df[(final_df["sa2_name"].eq(selected_name)) & (final_df["year"].isin([2019, 2025]))]
    log(f"Requested: {name}; using: {selected_name}")
    log(format_df(comparison[["year", "sa2_name", "non_employing_count", "working_age_pop", "non_employing_per_1000_working_age", "change_abs_since_2019", "change_pct_since_2019"]]))

log_section("11. Established middle-ring SA2 2019 vs 2025 comparison")
established_requested = ["Brighton (Vic.)", "Box Hill", "Camberwell"]
for name in established_requested:
    if name in set(final_df["sa2_name"]):
        selected_name = name
    else:
        matches = sorted([candidate for candidate in final_df["sa2_name"].dropna().unique() if name in candidate])
        selected_name = matches[0] if matches else None
    if selected_name is None:
        log(f"No match found for {name}")
        continue
    comparison = final_df[(final_df["sa2_name"].eq(selected_name)) & (final_df["year"].isin([2019, 2025]))]
    log(f"Requested: {name}; using: {selected_name}")
    log(format_df(comparison[["year", "sa2_name", "non_employing_count", "working_age_pop", "non_employing_per_1000_working_age", "change_abs_since_2019", "change_pct_since_2019"]]))

log_section("12. 2025 primary-view distribution summary")
summary = frame_2025_primary["non_employing_per_1000_working_age"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
summary = summary.rename(index={"10%": "p10", "25%": "p25", "50%": "p50", "75%": "p75", "90%": "p90", "95%": "p95"})
log(summary.loc[["min", "p10", "p25", "p50", "p75", "p90", "p95", "max"]].to_string())

final_df.to_csv(MAP3_CSV_OUTPUT, index=False)
MAP3_LOG_OUTPUT.write_text("\n".join(log_lines) + "\n", encoding="utf-8")

log_section("Output verification")
log(f"Wrote CSV: {MAP3_CSV_OUTPUT} ({MAP3_CSV_OUTPUT.stat().st_size:,} bytes)")
log(f"Wrote log: {MAP3_LOG_OUTPUT} ({MAP3_LOG_OUTPUT.stat().st_size:,} bytes)")
read_back = pd.read_csv(MAP3_CSV_OUTPUT)
log(f"CSV read-back rows: {len(read_back):,}; columns: {list(read_back.columns)}")
required_headers = [
    "=== 1. Total row count of final table ===",
    "=== 2. Count of rows per year where include_in_primary_view is True ===",
    "=== 12. 2025 primary-view distribution summary ===",
]
log_text = MAP3_LOG_OUTPUT.read_text(encoding="utf-8")
missing_headers = [header for header in required_headers if header not in log_text]
if missing_headers:
    raise ValueError(f"Log file missing required section headers: {missing_headers}")
log("Map 3 data pipeline completed successfully.")
MAP3_LOG_OUTPUT.write_text("\n".join(log_lines) + "\n", encoding="utf-8")

## Build Map Outputs

Builds a self-contained Plotly choropleth HTML and a static 2025 PNG preview from the analysis CSV (`data/processed/map3_home_business_density.csv`) joined to the SA2 polygons in `data/processed/melbourne.gpkg`. Two views (Residential Melbourne default, All Melbourne incl. CBD alternate) and a year animation slider 2019 to 2025.

**View toggle approach:** two-figure HTML scaffold. We build two independent `go.Figure` objects, one per view, each with its own 7-step year slider. The two figures are stacked into a single HTML wrapper with a `<select>` control that flips `display:none/block` between them. Per-figure slider state is preserved across toggles, and any (view, year) combination is reachable.

**Self-containment:** Plotly.js is inlined into the HTML (`include_plotlyjs=True`), so the page renders without a CDN. The Carto/OpenStreetMap basemap tiles are still fetched online at view time — removing that dependency would require dropping the basemap (`mapbox_style="white-bg"`), which strips the geographic context, so we keep the tiled basemap and accept tiles as the single remaining online dependency.

Outputs:

- `outputs/maps/home_business_density.html`
- `outputs/maps/home_business_density_2025_preview.png`

### Imports and Paths

This cell loads the visualisation dependencies and resolves repo-relative paths. The visualisation reads the analysis CSV and `melbourne.gpkg`, then writes map deliverables under `outputs/maps/`.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go

CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not find the repo root containing the data/ directory.")

DATA_PATH = REPO_ROOT / "data" / "processed" / "map3_home_business_density.csv"
GPKG_PATH = REPO_ROOT / "data" / "processed" / "melbourne.gpkg"
OUT_DIR = REPO_ROOT / "outputs" / "maps"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH: {DATA_PATH} (exists={DATA_PATH.exists()})")
print(f"GPKG_PATH: {GPKG_PATH} (exists={GPKG_PATH.exists()})")
print(f"OUT_DIR:   {OUT_DIR}")

### Load Metrics, SA2 Geometry, and Council Names

This cell joins the tidy SA2-year metrics to the SA2 polygon layer and attaches council names using the same centroid-in-polygon approach as `infra_map.ipynb`. The SA2 polygons are simplified before conversion to GeoJSON so the final HTML remains practical to share while preserving visible map detail at the intended zoom.

In [ ]:
# Load CSV (sa2_code as string preserves any leading zeros)
df = pd.read_csv(DATA_PATH, dtype={"sa2_code": str})

# Load SA2 polygon layer and reproject to WGS84 for Plotly choroplethmapbox
sa2_gdf = gpd.read_file(GPKG_PATH, layer="greater_melb_sa2")
sa2_gdf = sa2_gdf[["SA2_CODE_2021", "SA2_NAME_2021", "geometry"]].copy()
sa2_gdf["SA2_CODE_2021"] = sa2_gdf["SA2_CODE_2021"].astype(str)
sa2_gdf = sa2_gdf.to_crs(epsg=4326)

# Simplify polygon geometry to keep the embedded GeoJSON small. Plotly embeds the
# full geojson inside every trace, so unsimplified ASGS polygons inflate the HTML
# to ~120 MB. A 1e-4 degree tolerance (~11 m at this latitude) is well below
# visible detail at the default zoom of 8.3.
SIMPLIFY_TOLERANCE_DEG = 1e-4
sa2_gdf["geometry"] = sa2_gdf.geometry.simplify(SIMPLIFY_TOLERANCE_DEG, preserve_topology=True)

# LGA layer (same as infra_map.ipynb), keep name column only
lga = gpd.read_file(GPKG_PATH, layer="greater_melb_local_gov_2025_any_intersect")
lga = lga[["LGA_NAME_2025", "geometry"]].to_crs(sa2_gdf.crs)

# SA2 centroid -> LGA spatial join in a projected CRS for accurate centroids
sa2_centroids = sa2_gdf.to_crs(7855).copy()
sa2_centroids["geometry"] = sa2_centroids.geometry.centroid
sa2_centroids = sa2_centroids.to_crs(sa2_gdf.crs)

joined = gpd.sjoin(
    sa2_centroids[["SA2_CODE_2021", "geometry"]],
    lga,
    how="left",
    predicate="within",
)
sa2_council = joined[["SA2_CODE_2021", "LGA_NAME_2025"]].rename(columns={"LGA_NAME_2025": "council"})

# Attach council to the metric dataframe via sa2_code <-> SA2_CODE_2021
df_rows_before = len(df)
df = df.merge(sa2_council, left_on="sa2_code", right_on="SA2_CODE_2021", how="left").drop(columns=["SA2_CODE_2021"])
assert len(df) == df_rows_before, "Row count changed during council merge"

# GeoJSON for Plotly (featureidkey points to properties.SA2_CODE_2021)
sa2_geojson = json.loads(sa2_gdf[["SA2_CODE_2021", "geometry"]].to_json())

print(f"CSV rows after council merge: {len(df):,} (expected {df_rows_before:,})")
print(f"SA2/year rows missing council:  {df['council'].isna().sum():,}")
print(f"Unique SA2s missing council:    {df.loc[df['council'].isna(), 'sa2_code'].nunique():,}")
print(f"SA2 polygons in GeoJSON:        {len(sa2_geojson['features']):,}")
print(f"Unique LGAs joined:             {df['council'].nunique():,}")

### Colour Scale Caps

The map uses the 2025 95th percentile as the colour cap for each view. This keeps a small number of high-density SA2s from flattening colour contrast across the rest of Melbourne while still preserving the true metric values in hover tooltips.

In [ ]:
# 95th percentile of the 2025 metric within each view -> per-view colour cap
RESIDENTIAL_CMAX = float(
    df.loc[(df["year"] == 2025) & df["include_in_primary_view"],
           "non_employing_per_1000_working_age"].quantile(0.95)
)
ALL_MELB_CMAX = float(
    df.loc[(df["year"] == 2025) & df["include_in_alternate_view"],
           "non_employing_per_1000_working_age"].quantile(0.95)
)

print(f"RESIDENTIAL_CMAX (p95, 2025 primary view):   {RESIDENTIAL_CMAX:.2f}")
print(f"ALL_MELB_CMAX    (p95, 2025 alternate view): {ALL_MELB_CMAX:.2f}")

### Build Choropleth Traces

This cell creates one Plotly choropleth trace for every `(view, year)` combination. The Residential Melbourne view uses `include_in_primary_view`; the All Melbourne view uses `include_in_alternate_view`. Hover data is written in plain language and carries the SA2 name, council, year, working-age denominator, non-employing business count, density metric, and change-since-2019 fields.

In [ ]:
# 14 traces total: 7 per view, grouped by view so each view can become its own figure.
VIEWS = [
    ("residential", "Residential Melbourne",   "include_in_primary_view",   RESIDENTIAL_CMAX),
    ("all_melb",    "All Melbourne (incl. CBD)", "include_in_alternate_view", ALL_MELB_CMAX),
]
YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
DEFAULT_YEAR = 2025

HOVER_TEMPLATE = (
    "<b>%{customdata[0]}</b><br>"
    "Council: %{customdata[1]}<br>"
    "Year: %{customdata[2]:.0f}<br><br>"
    "Working-age residents (15-64): %{customdata[3]:,.0f}<br>"
    "Non-employing businesses: %{customdata[4]:,.0f}<br>"
    "Home-based business density: %{customdata[5]:.1f} per 1,000 working-age residents<br>"
    "Change since 2019: %{customdata[6]:+.1f} per 1,000 (%{customdata[7]:+.1f}%)"
    "<extra></extra>"
)
COLORBAR_TITLE = "Non-employing<br>per 1,000<br>working-age"

view_traces = {}
for view_key, _, include_col, cmax in VIEWS:
    traces = []
    for year in YEARS:
        subset = df[(df["year"] == year) & df[include_col]].copy()
        subset["council"] = subset["council"].fillna("Unknown")
        is_default_year = (year == DEFAULT_YEAR)

        traces.append(go.Choroplethmapbox(
            geojson=sa2_geojson,
            featureidkey="properties.SA2_CODE_2021",
            locations=subset["sa2_code"],
            z=subset["non_employing_per_1000_working_age"],
            colorscale="Viridis_r",
            zmin=0,
            zmax=cmax,
            marker_line_width=0.3,
            marker_line_color="white",
            marker_opacity=0.85,
            customdata=subset[[
                "sa2_name", "council", "year", "working_age_pop", "non_employing_count",
                "non_employing_per_1000_working_age",
                "change_abs_since_2019", "change_pct_since_2019",
            ]].values,
            hovertemplate=HOVER_TEMPLATE,
            colorbar=dict(title=COLORBAR_TITLE),
            showscale=is_default_year,
            name=f"{view_key}_{year}",
            visible=is_default_year,
        ))
    view_traces[view_key] = traces

total_traces = sum(len(t) for t in view_traces.values())
print(f"Built {total_traces} traces grouped into {len(view_traces)} views x {len(YEARS)} years")
for view_key in view_traces:
    print(f"  {view_key}: {len(view_traces[view_key])} traces, default visible = {view_key}_{DEFAULT_YEAR}")

### Build One Figure Per View

Each view gets its own Plotly figure, year slider, and Play/Pause controls. Keeping the views separate avoids Plotly state conflicts between a view toggle and a year animation; the HTML wrapper in the next cell simply shows or hides the relevant figure.

In [ ]:
# Build one go.Figure per view, each with its own 7-step year slider and Play/Pause controls.
# No view-toggle control lives inside the figure - that lives in the surrounding HTML scaffold.
def _visibility_for_year_index(view_traces_list, active_index):
    visible = [False] * len(view_traces_list)
    showscale = [False] * len(view_traces_list)
    visible[active_index] = True
    showscale[active_index] = True
    return visible, showscale


def _year_frames_for_view(view_traces_list):
    frames = []
    trace_indexes = list(range(len(view_traces_list)))
    for i, year in enumerate(YEARS):
        visible, showscale = _visibility_for_year_index(view_traces_list, i)
        frame_data = [
            go.Choroplethmapbox(visible=is_visible, showscale=show_scale)
            for is_visible, show_scale in zip(visible, showscale)
        ]
        frames.append(go.Frame(name=str(year), data=frame_data, traces=trace_indexes))
    return frames


def _year_slider_for_view(view_traces_list):
    steps = []
    for i, year in enumerate(YEARS):
        steps.append({
            "label": str(year),
            "method": "animate",
            "args": [[str(year)], {
                "mode": "immediate",
                "frame": {"duration": 0, "redraw": True},
                "transition": {"duration": 0},
            }],
        })
    return {
        "active": YEARS.index(DEFAULT_YEAR),
        "currentvalue": {"prefix": "Year: ", "font": {"size": 14}},
        "pad": {"t": 30, "b": 10},
        "len": 0.85,
        "x": 0.075,
        "y": 0.02,
        "steps": steps,
    }


ANIMATION_BUTTONS = [{
    "type": "buttons",
    "direction": "left",
    "showactive": False,
    "x": 0.075,
    "y": 0.10,
    "xanchor": "left",
    "yanchor": "bottom",
    "pad": {"t": 4, "r": 10},
    "buttons": [
        {
            "label": "Play",
            "method": "animate",
            "args": [None, {
                "fromcurrent": True,
                "mode": "immediate",
                "frame": {"duration": 900, "redraw": True},
                "transition": {"duration": 300},
            }],
        },
        {
            "label": "Pause",
            "method": "animate",
            "args": [[None], {
                "mode": "immediate",
                "frame": {"duration": 0, "redraw": False},
                "transition": {"duration": 0},
            }],
        },
    ],
}]

view_figures = {}
for view_key, view_label, _, _ in VIEWS:
    fig = go.Figure(data=view_traces[view_key], frames=_year_frames_for_view(view_traces[view_key]))
    fig.update_layout(
        mapbox_style="carto-positron",
        mapbox_center={"lat": -37.85, "lon": 144.95},
        mapbox_zoom=8.3,
        margin={"l": 0, "r": 0, "t": 30, "b": 0},
        height=760,
        sliders=[_year_slider_for_view(view_traces[view_key])],
        updatemenus=ANIMATION_BUTTONS,
    )
    view_figures[view_key] = fig

print(f"Built {len(view_figures)} figures, each with {len(YEARS)} year slider steps and Play/Pause controls")
for view_key, fig in view_figures.items():
    print(f"  {view_key}: {len(fig.data)} traces, {len(fig.frames)} frames, default year = {DEFAULT_YEAR}")

### Write Interactive HTML

This cell wraps the two Plotly figures in a small HTML scaffold with a view selector. Plotly.js is inlined once, so the page does not rely on a Plotly CDN; the Carto/OpenStreetMap basemap tiles still load from the internet when the map is viewed.

In [ ]:
HTML_OUT = OUT_DIR / "home_business_density.html"

ATTRIBUTION = (
    "Source: ABS Counts of Australian Businesses (CABEE) 2017-2025 releases; "
    "ABS Estimated Resident Population 2024. Greater Melbourne SA2s with working-age "
    "population &ge;500. Seven CBD/Southbank/industrial-estate SA2s flagged separately "
    "for the view toggle."
)
PAGE_TITLE = "Where is Melbourne&rsquo;s home-based economy growing?"
PAGE_SUBTITLE = "Non-employing businesses per 1,000 working-age residents, by SA2 (2019&ndash;2025)"

# Stable div ids so the toggle JS can target them.
view_div_ids = {view_key: f"map-{view_key}" for view_key, *_ in VIEWS}

# Render each figure as a fragment. plotly.js is inlined exactly once on the first
# figure; subsequent figures reuse the same loaded library.
fig_fragments = {}
for i, (view_key, _, _, _) in enumerate(VIEWS):
    include_plotlyjs = True if i == 0 else False
    fig_fragments[view_key] = view_figures[view_key].to_html(
        full_html=False,
        include_plotlyjs=include_plotlyjs,
        div_id=view_div_ids[view_key],
        config={"responsive": True, "displaylogo": False},
    )

view_options_html = "\n".join(
    f'        <option value="{view_key}"{" selected" if i == 0 else ""}>{view_label}</option>'
    for i, (view_key, view_label, _, _) in enumerate(VIEWS)
)
default_view_key = VIEWS[0][0]
view_panel_html = "\n".join(
    f'    <div class="map-panel" id="panel-{view_key}" '
    f'style="display: {"block" if view_key == default_view_key else "none"};">\n'
    f'      {fig_fragments[view_key]}\n'
    f'    </div>'
    for view_key, *_ in VIEWS
)
panel_ids_js = ", ".join(f'"panel-{view_key}"' for view_key, *_ in VIEWS)
map_ids_js = ", ".join(f'"{view_div_ids[view_key]}"' for view_key, *_ in VIEWS)

scaffold = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
<meta charset=\"utf-8\" />
<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
<title>{PAGE_TITLE}</title>
<style>
  body {{ margin: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #222; background: #fff; }}
  header {{ padding: 16px 24px 6px; }}
  header h1 {{ margin: 0 0 4px; font-size: 20px; }}
  header .subtitle {{ margin: 0; color: #555; font-size: 13px; }}
  .controls {{ padding: 8px 24px 4px; display: flex; align-items: center; gap: 10px; font-size: 13px; }}
  .controls label {{ color: #444; }}
  .controls select {{ font-size: 13px; padding: 4px 8px; border: 1px solid #ccc; border-radius: 4px; background: #fafafa; }}
  .map-panel {{ width: 100%; }}
  footer {{ padding: 6px 24px 14px; font-size: 11px; color: #555; line-height: 1.4; }}
</style>
</head>
<body>
<header>
  <h1>{PAGE_TITLE}</h1>
  <p class=\"subtitle\">{PAGE_SUBTITLE}</p>
</header>
<div class=\"controls\">
  <label for=\"view-select\">View:</label>
  <select id=\"view-select\">
{view_options_html}
  </select>
</div>
{view_panel_html}
<footer>{ATTRIBUTION}</footer>
<script>
  (function () {{
    var panelIds = [{panel_ids_js}];
    var mapIds   = [{map_ids_js}];
    var select   = document.getElementById("view-select");

    function showView(viewKey) {{
      for (var i = 0; i < panelIds.length; i++) {{
        var panel = document.getElementById(panelIds[i]);
        var isActive = panelIds[i] === ("panel-" + viewKey);
        panel.style.display = isActive ? "block" : "none";
        if (isActive && window.Plotly) {{
          var mapDiv = document.getElementById(mapIds[i]);
          if (mapDiv) {{ window.Plotly.Plots.resize(mapDiv); }}
        }}
      }}
    }}

    select.addEventListener("change", function (e) {{ showView(e.target.value); }});
  }})();
</script>
</body>
</html>
"""

HTML_OUT.write_text(scaffold, encoding="utf-8")
print(f"Wrote HTML: {HTML_OUT} ({HTML_OUT.stat().st_size:,} bytes)")

### Write Static 2025 Preview PNG

This cell exports a separate static figure for the default 2025 Residential Melbourne frame. The PNG is intended for slide decks, quick review, and situations where opening the interactive HTML is inconvenient. PNG export requires Kaleido and a working local Chrome/Chrome-for-Testing install.

In [ ]:
PNG_OUT = OUT_DIR / "home_business_density_2025_preview.png"

# Standalone figure for the PNG so the interactive figures are not mutated.
png_year_idx = YEARS.index(2025)
png_source = view_traces["residential"][png_year_idx]
png_fig = go.Figure(data=[go.Choroplethmapbox(
    geojson=sa2_geojson,
    featureidkey="properties.SA2_CODE_2021",
    locations=png_source.locations,
    z=png_source.z,
    colorscale=png_source.colorscale,
    zmin=png_source.zmin,
    zmax=png_source.zmax,
    marker_line_width=png_source.marker.line.width,
    marker_line_color=png_source.marker.line.color,
    marker_opacity=png_source.marker.opacity,
    customdata=png_source.customdata,
    hovertemplate=png_source.hovertemplate,
    colorbar=dict(title=COLORBAR_TITLE),
    showscale=True,
    name="residential_2025",
)])
png_fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_center={"lat": -37.85, "lon": 144.95},
    mapbox_zoom=8.3,
    margin={"l": 0, "r": 0, "t": 80, "b": 40},
    title={
        "text": "<b>Where is Melbourne's home-based economy growing?</b><br>"
                "<sub>Non-employing businesses per 1,000 working-age residents, by SA2 (2025)</sub>",
        "x": 0.5,
        "xanchor": "center",
    },
    annotations=[{
        "text": (
            "Source: ABS Counts of Australian Businesses (CABEE) 2017-2025 releases; "
            "ABS Estimated Resident Population 2024. Greater Melbourne SA2s with "
            "working-age population >=500. Seven CBD/Southbank/industrial-estate "
            "SA2s flagged separately for view toggle."
        ),
        "showarrow": False,
        "xref": "paper", "yref": "paper",
        "x": 0.5, "y": -0.02,
        "xanchor": "center", "yanchor": "top",
        "font": {"size": 10, "color": "#555"},
    }],
)
png_fig.write_image(PNG_OUT, width=1600, height=1000, scale=2)
print(f"Wrote PNG:  {PNG_OUT} ({PNG_OUT.stat().st_size:,} bytes)")

### Final Visualisation Sanity Check

This final cell prints the key checks: output file sizes, colour caps, rendered SA2 counts for both 2025 views, council-join completeness, and confirmation that all view-year combinations are reachable with sliders and Play/Pause controls.

In [ ]:
html_size_mb = HTML_OUT.stat().st_size / (1024 * 1024)
png_size_kb = PNG_OUT.stat().st_size / 1024

residential_2025_count = int(((df["year"] == 2025) & df["include_in_primary_view"]).sum())
all_melb_2025_count = int(((df["year"] == 2025) & df["include_in_alternate_view"]).sum())
missing_council = int(df["council"].isna().sum())

print("=== Map output sanity check ===")
print(f"HTML output: {HTML_OUT.name} ({html_size_mb:.2f} MB, plotly.js inlined)")
print(f"PNG output:  {PNG_OUT.name} ({png_size_kb:.0f} KB)")
print(f"RESIDENTIAL_CMAX (p95):   {RESIDENTIAL_CMAX:.1f}")
print(f"ALL_MELB_CMAX    (p95):   {ALL_MELB_CMAX:.1f}")
print(f"SA2s rendered, residential 2025 frame: {residential_2025_count}")
print(f"SA2s rendered, all-melb 2025 frame:    {all_melb_2025_count}")
print(f"SA2/year rows missing council join:    {missing_council}"
      f"{' (OK)' if missing_council == 0 else ' (CHECK)'}")
print(f"View x year combinations reachable:    {len(VIEWS) * len(YEARS)} "
      f"({len(VIEWS)} views x {len(YEARS)} years, sliders and Play/Pause controls enabled)")